In [ ]:
import tensorflow as tf
import numpy as np

print(tf.__version__)

## Dataset

In [ ]:
texts = [
    "i love this movie",
    "this movie is amazing",
    "i hate this movie",
    "this movie is terrible",
    "fantastic film",
    "worst movie ever"
]

labels = [1, 1, 0, 0, 1, 0]

## Text Vectorization

In [ ]:
max_tokens = 1000
sequence_length = 6

vectorizer = tf.keras.layers.TextVectorization(
    max_tokens=max_tokens,
    output_mode="int",
    output_sequence_length=sequence_length
)

vectorizer.adapt(texts)
print(vectorizer.get_vocabulary())

## Ubah kalimat jadi token

In [ ]:
x = vectorizer(texts)

print(x.numpy())

## Embedding

In [ ]:
embedding_dim = 8

embedding = tf.keras.layers.Embedding(
    input_dim=max_tokens,
    output_dim=embedding_dim
)

embedded = embedding(x)
print(embedded.shape)
print(embedded[0].numpy())

## Membuat query, key, value

In [ ]:
dense_q = tf.keras.layers.Dense(embedding_dim)
dense_k = tf.keras.layers.Dense(embedding_dim)
dense_v = tf.keras.layers.Dense(embedding_dim)

Q = dense_q(embedded)
K = dense_k(embedded)
V = dense_v(embedded)

print(Q.shape)
print(K.shape)
print(V.shape)

## Hitung Attention Score

In [ ]:
score = tf.matmul(
    Q,
    K,
    transpose_b=True
)
print(score.shape)

In [ ]:
print(score[0].numpy())

## Scalling (dibagi dengan akar(Dk))

In [ ]:
dk = tf.cast(tf.shape(K)[-1], tf.float32)

scaled_score = score / tf.math.sqrt(dk)

## Softmax

In [ ]:
attention_weight = tf.nn.softmax(
    scaled_score,
    axis=-1
)
print(attention_weight[0].numpy())

In [ ]:
print(tf.reduce_sum(attention_weight[0], axis=-1).numpy())

## Weighted Sum

In [ ]:
output = tf.matmul(
    attention_weight,
    V
)
print(output.shape)
print(output[0].numpy())

## Buat layer seflAttention

In [ ]:
import tensorflow as tf

class SelfAttention(tf.keras.layers.Layer):
    def __init__(self, embed_dim):
        super().__init__()
        self.embed_dim = embed_dim
        self.Wq = tf.keras.layers.Dense(embed_dim)
        self.Wk = tf.keras.layers.Dense(embed_dim)
        self.Wv = tf.keras.layers.Dense(embed_dim)

    def call(self, x):
        Q = self.Wq(x)
        K = self.Wk(x)
        V = self.Wv(x)
        score = tf.matmul(Q, K, transpose_b=True)
        dk = tf.cast(tf.shape(K)[-1], tf.float32)
        score = score / tf.math.sqrt(dk)
        attention = tf.nn.softmax(score, axis=-1)
        output = tf.matmul(attention, V)
        return output

In [ ]:
attention = SelfAttention(8)
hasil = attention(embedded)
print(hasil.shape)

## Feed Forward Network (FFN)

In [ ]:
class FeedForward(tf.keras.layers.Layer):
    def __init__(self, embed_dim, ff_dim):
        super().__init__()

        self.dense1 = tf.keras.layers.Dense(
            ff_dim,
            activation="relu"
        )

        self.dense2 = tf.keras.layers.Dense(
            embed_dim
        )

    def call(self, x):
        x = self.dense1(x)
        x = self.dense2(x)
        return x

In [ ]:
ffn = FeedForward(
    embed_dim=8,
    ff_dim=32
)

hasil = ffn(embedded)

print(hasil.shape)

## Residual Connection

In [ ]:
x = embedded

attention_output = attention(x)

residual = x + attention_output

## Layer Normalization

In [ ]:
layer_norm = tf.keras.layers.LayerNormalization()

output = layer_norm(residual)
print(output.shape)

## FFN + Residual Lagi

In [ ]:
ff_output = ffn(output)

output = tf.keras.layers.Add()(
    [output, ff_output]
)

output = tf.keras.layers.LayerNormalization()(output)

## Gabungkan Jadi Encoder

In [ ]:
class Encoder(tf.keras.layers.Layer):
    def __init__(self,
                 embed_dim,
                 ff_dim):
        super().__init__()
        self.attention = SelfAttention(embed_dim)
        self.norm1 = tf.keras.layers.LayerNormalization()
        self.ffn = FeedForward(
            embed_dim,
            ff_dim
        )
        self.norm2 = tf.keras.layers.LayerNormalization()

    def call(self, x):
        attention_output = self.attention(x)
        x = x + attention_output
        x = self.norm1(x)
        ff_output = self.ffn(x)
        x = x + ff_output
        x = self.norm2(x)
        return x

## Coba Encoder

In [ ]:
encoder = Encoder(
    embed_dim=8,
    ff_dim=32
)

hasil = encoder(embedded)

print(hasil.shape)

## Dataset v2

In [ ]:
import tensorflow as tf

texts = [
    "i love this movie",
    "this movie is amazing",
    "fantastic film",
    "this is wonderful",

    "i hate this movie",
    "this movie is terrible",
    "worst movie ever",
    "this is awful"
]

labels = [
    1,
    1,
    1,
    1,

    0,
    0,
    0,
    0
]

## Text Vectorization

In [ ]:
max_tokens = 1000
sequence_length = 6

vectorizer = tf.keras.layers.TextVectorization(
    max_tokens=max_tokens,
    output_mode="int",
    output_sequence_length=sequence_length
)

vectorizer.adapt(texts)

## Embedding

In [ ]:
embedding_dim = 32

embedding = tf.keras.layers.Embedding(
    input_dim=max_tokens,
    output_dim=embedding_dim
)

## Encoder

## Banugn Model

In [ ]:
inputs = tf.keras.Input(shape=(1,), dtype=tf.string)

x = vectorizer(inputs)
x = embedding(x)

encoder = Encoder(
    embed_dim=32,
    ff_dim=64
)

x = encoder(x)
x = tf.keras.layers.GlobalAveragePooling1D()(x)
x = tf.keras.layers.Dense(32,activation="relu")(x)
outputs = tf.keras.layers.Dense(2,activation="softmax")(x)
model = tf.keras.Model(inputs, outputs)

## Compile

In [ ]:
model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

## Lihat Arsitektur

In [ ]:
model.summary()

## Training

In [ ]:
x_train = tf.constant(texts, dtype=tf.string)
y_train = tf.constant(labels, dtype=tf.int32)

model.fit(
    x_train,
    y_train,
    epochs=30
)

## Evaluate

In [ ]:
loss, acc = model.evaluate(
    x_train,
    y_train
)

print(loss)
print(acc)

## Predict

In [ ]:
test = tf.constant([
    "this movie is fantastic",
    "this movie is terrible",
    "i love this movie",
    "worst movie ever"
], dtype=tf.string)

pred = model.predict(test)

print(pred)
print()

classes = ["Negative", "Positive"]

for kalimat, probabilitas in zip(test.numpy(), pred):

    idx = tf.argmax(probabilitas).numpy()

    print(f"Kalimat : {kalimat.decode()}")
    print(f"Prediksi: {classes[idx]}")
    print(f"Confidence: {probabilitas[idx]*100:.2f}%")
    print("-"*40)